# Lesson 6 - Wrapping a Smolagents Agent into an ACP Server

You will now create another ACP agent built with Smolagents. This second agent will be able to search the web to handle health based questions for patients.  You will wrap the agent definition in an ACP server similarly to how you did with the first RAG agent.

## 6.1. Wrap the Agent in ACP Server

To define your agent, you will use [CodeAgent](https://smolagents.org/docs/agents-guided-tour/) from the Smolagents library. This type of agent writes and executes Python code at every step. For this agent, you will use two tools provided by Smolagents:

- DuckDuckGoSearchTool: performs a web search using DuckDuckGo browser
- VisitWebpageTool: can visit and extract content from web pages

The agent is wrapped in an ACP server using `@server.agent()` decorator. The server will also run locally using a different port number: 8001. Run the following cell to save the agent as `smolagents_server.py` under `my_acp_project`.

In [1]:
!pip install duckduckgo-search --quiet


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
INFERENCE_SERVER_URL = "http://localhost:8989"
MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
API_KEY= "alanliuxiang"

In [3]:
# %%writefile ../my_acp_project/smolagents_server.py
from collections.abc import AsyncGenerator
from acp_sdk.models import Message, MessagePart
from acp_sdk.server import Context, RunYield, RunYieldResume, Server
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel, VisitWebpageTool
import logging 
from dotenv import load_dotenv

import nest_asyncio
nest_asyncio.apply()

load_dotenv() 

server = Server()

from smolagents import OpenAIServerModel

# Configure the model to use LM Studio's local API endpoint
model = OpenAIServerModel(
    model_id= "ibm-granite/granite-3.3-2b-instruct",  # This can be any name, LM Studio will use whatever model you have loaded
    api_base= f"{INFERENCE_SERVER_URL}/v1",  # Default LM Studio API endpoint
    api_key=API_KEY # LM Studio doesn't require an API key by default
)


@server.agent()
async def health_agent(input: list[Message], context: Context) -> AsyncGenerator[RunYield, RunYieldResume]:
    "This is a CodeAgent which supports the hospital to handle health based questions for patients. Current or prospective patients can use it to find answers about their health and hospital treatments."
    agent = CodeAgent(tools=[DuckDuckGoSearchTool(), VisitWebpageTool()], model=model)

    prompt = input[0].parts[0].content
    response = agent.run(prompt)

    yield Message(parts=[MessagePart(content=str(response))])


if __name__ == "__main__":
    server.run(port=8003)

INFO:     Started server process [4052]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8003 (Press CTRL+C to quit)
INFO:     Run started
ERROR:    Run failed
Traceback (most recent call last):
  File "/opt/app-root/lib64/python3.11/site-packages/smolagents/default_tools.py", line 130, in __init__
    from ddgs import DDGS
ModuleNotFoundError: No module named 'ddgs'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/app-root/lib64/python3.11/site-packages/acp_sdk/server/executor.py", line 210, in _execute
    raise next
  File "/opt/app-root/lib64/python3.11/site-packages/acp_sdk/server/executor.py", line 291, in _run_async_gen
    value = await context.yield_async(await gen.asend(value))
                                      ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/app-root/lib64/python3.11/site-packages/acp_sdk/server/agent.py", line 102, in 

INFO:     127.0.0.1:32954 - "POST /runs HTTP/1.1" 200 OK


INFO:     Run started
ERROR:    Task was destroyed but it is pending!
task: <Task pending name='Task-14' coro=<Executor._run_async_gen() running at /opt/app-root/lib64/python3.11/site-packages/acp_sdk/server/executor.py:295> wait_for=<Future pending cb=[Task.__wakeup()]>>


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Do I need rehabilitation after a shoulder reconstruction?                                                       │
│                                                                                                                 │
╰─ OpenAIServerModel - ibm-granite/granite-3.3-2b-instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 19: Duration 6.61 seconds]

ERROR:    Run failed
Traceback (most recent call last):
  File "/opt/app-root/lib64/python3.11/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/opt/app-root/lib64/python3.11/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/app-root/lib64/python3.11/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/opt/app-root/lib64/python3.11/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/app-root/lib64/python3.11/site-packages/httpcore/_sync/connection.py", line 101, in handle_request
    raise exc
  File "/opt/app-root/lib64/python3.11/site-packages/httpcore/_sync/connection.py", line 78, in handle_request
    stream = self._connect(request)
  

INFO:     127.0.0.1:38596 - "POST /runs HTTP/1.1" 200 OK


ERROR:    Task was destroyed but it is pending!
task: <Task pending name='Task-21' coro=<Executor._run_async_gen() done, defined at /opt/app-root/lib64/python3.11/site-packages/acp_sdk/server/executor.py:286> wait_for=<Future pending cb=[Task.__wakeup()]>>


RuntimeError: Event loop stopped before Future completed.

## 6.2. Run the Hospital ACP Server

Now to activate your configured ACP agent, you also need to run your agent from another terminal.

- Open the second terminal by running the cell below;
- Type `uv run smolagents_server.py` to run the server and activate your ACP agent.

Please see note below if you want to replicate the work locally on your machine.

In [ ]:
# from IPython.display import IFrame
# import os
# url = os.environ.get('DLAI_LOCAL_URL').format(port=8888)
# IFrame(f"{url}terminals/2", width=800, height=600)

If you see this warning: 
`WARNING: Can not reach server, check if running on http://127.0.0.1:8333 : Request failed after 5 retries`
you can ignore it. You'll learn later in another lesson about the BeeAI platform, which a registry you can use to manage and discover agents. If the platform is installed, it runs by default on port 8333. The ACP servers are configured to automatically connect to the platform. Since the platform is not installed in this environment, the ACP server will generate a warning.

**Note: How to update the `my_acp_project` locally on your machine so you can run the second server?**
- cd `my_acp_project`
- `uv add smolagents duckduckgo-search markdownify requests`

## 6.3. Resources

- [Smolagents doc](https://smolagents.org/docs/smolagent-docs/)
- [Short course: Building Code Agents with Hugging Face Smolagents ](https://www.deeplearning.ai/short-courses/building-code-agents-with-hugging-face-smolagents/)
- [Same code using a local open source model: `ollama_chat/qwen2.5:14b`](https://github.com/nicknochnack/ACPWalkthrough/blob/main/4.%20smolagents%20ACP.py)

<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>To access the <code>my_acp_project</code> folder:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. 

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

</div>